# Biohub model56 — top-five association production graph

Checksum-pinned primary detector, top-five parent candidates, native ILP, and fixed geometry-only post-processing. No training labels are read at inference.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

import torch

COMPETITION = "biohub-cell-tracking-during-development"
EXPECTED_WEIGHT_SHA256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
EXPECTED_PREDICTOR_SHA256 = "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9"
DET_THRESHOLD = 0.965
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.0
ILP_DISAPPEARANCE_WEIGHT = 2.0
ILP_DIVISION_WEIGHT = 1.2

INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
COMPETITION_CANDIDATES = [
    INPUT_ROOT / "competitions" / COMPETITION,
    INPUT_ROOT / COMPETITION,
]
COMPETITION_DIR = next((path for path in COMPETITION_CANDIDATES if path.is_dir()), None)
if COMPETITION_DIR is None:
    raise FileNotFoundError({"competition_candidates": [str(path) for path in COMPETITION_CANDIDATES]})
TEST_DIR = COMPETITION_DIR / "test"
test_stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
if not test_stems:
    raise RuntimeError(f"No test Zarrs found under {TEST_DIR}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; launch this notebook with --accelerator NvidiaTeslaT4")
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT != 2:
    raise RuntimeError(f"Expected Kaggle T4 x2, found {GPU_COUNT} CUDA device(s)")
GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)]
if not all("T4" in name for name in GPU_NAMES):
    raise RuntimeError(f"Expected Tesla T4 devices, found {GPU_NAMES}")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def artifact_weight_sha256(root: Path) -> str:
    manifest_path = root / "ARTIFACT_MANIFEST.json"
    if not manifest_path.is_file():
        return ""
    try:
        payload = json.loads(manifest_path.read_text())
    except Exception:
        return ""
    direct = str(payload.get("model", {}).get("weight_sha256", ""))
    nested = str(
        payload.get("models", {})
        .get("unet_transformer", {})
        .get("weight_sha256", "")
    )
    return direct or nested


def candidate_artifact_roots() -> list[Path]:
    preferred = [
        INPUT_ROOT / "biohub-tracking-support-pack-50ep-v1",
        INPUT_ROOT / "datasets" / "pilkwang" / "biohub-tracking-support-pack-50ep-v1",
    ]
    discovered: list[Path] = []
    if INPUT_ROOT.is_dir():
        for child in INPUT_ROOT.iterdir():
            if not child.is_dir():
                continue
            discovered.extend([child, child / child.name])
            try:
                discovered.extend(path for path in child.iterdir() if path.is_dir())
            except OSError:
                pass
    output: list[Path] = []
    seen: set[Path] = set()
    for path in [*preferred, *discovered]:
        if path in seen:
            continue
        seen.add(path)
        output.append(path)
    return output


ARTIFACTS = next(
    (
        root
        for root in candidate_artifact_roots()
        if artifact_weight_sha256(root) == EXPECTED_WEIGHT_SHA256
        and (root / "repo" / "scripts" / "predict_unet_transformer.py").is_file()
        and (root / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth").is_file()
    ),
    None,
)
if ARTIFACTS is None:
    raise FileNotFoundError(
        "Could not find the checksum-pinned primary support pack; attach "
        "pilkwang/biohub-tracking-support-pack-50ep-v1"
    )

# Install only packages absent or known to be incompatible.  NumPy, SciPy,
# and torch are intentionally left to Kaggle's coherent GPU image.
PACKAGE_SPECS = [
    "tracksdata", "zarr>=3.0.10,<4", "numcodecs>=0.13,<0.16", "blosc2",
    "geff>=1.1.3.1.1", "geff-spec<1.2", "ilpy>=0.5.1", "pyscipopt",
    "polars>=1.36", "polars-runtime-32", "dask", "imagecodecs", "pyarrow",
    "rustworkx>=0.17.1", "sqlalchemy>=2", "bidict", "psygnal", "pydantic",
    "pydantic-core", "annotated-types", "typing-inspection", "rich",
    "markdown-it-py", "pygments", "donfig", "google-crc32c", "networkx",
    "deprecated", "wrapt", "ndindex", "msgpack", "numexpr", "click",
    "cloudpickle", "fsspec", "partd", "locket", "toolz", "pyyaml",
    "scikit-image", "imageio", "pillow", "tifffile", "lazy-loader", "tqdm",
]
REQUIRED_MODULES = [
    "tracksdata", "zarr", "numcodecs", "blosc2", "geff", "geff_spec",
    "ilpy", "pyscipopt", "polars", "dask", "imagecodecs", "pyarrow",
    "rustworkx", "sqlalchemy", "skimage", "tqdm",
]


def dependencies_ready() -> bool:
    if any(importlib.util.find_spec(name) is None for name in REQUIRED_MODULES):
        return False
    try:
        # Inspect package metadata without importing native extensions.  If an
        # old Kaggle image needs the bundled upgrade, this avoids retaining a
        # stale polars/zarr module in sys.modules after pip replaces it.
        zarr_major = int(importlib.metadata.version("zarr").split(".", 1)[0])
        polars_parts = tuple(
            int(part) for part in importlib.metadata.version("polars").split(".")[:2]
        )
        return zarr_major >= 3 and polars_parts >= (1, 36)
    except Exception:
        return False


if not dependencies_ready():
    wheel_dir = ARTIFACTS / "wheels"
    if not wheel_dir.is_dir():
        raise FileNotFoundError(f"Offline wheel directory missing: {wheel_dir}")
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--find-links",
        str(wheel_dir),
        *PACKAGE_SPECS,
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout or "")[-3000:])
    print((result.stderr or "")[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    importlib.invalidate_caches()
if not dependencies_ready():
    failures = {}
    for module_name in REQUIRED_MODULES:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[module_name] = f"{type(exc).__name__}: {exc}"
    raise ImportError({"dependency_failures": failures})

REPO_DIR = WORKING_DIR / "topk_production_repo"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
weights_destination = REPO_DIR / "weights"
try:
    os.symlink(ARTIFACTS / "weights", weights_destination, target_is_directory=True)
except Exception:
    shutil.copytree(ARTIFACTS / "weights", weights_destination)

PREDICTOR_PATH = REPO_DIR / "scripts" / "predict_unet_transformer.py"
WEIGHT_PATH = REPO_DIR / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
actual_predictor_sha256 = sha256_file(PREDICTOR_PATH)
actual_weight_sha256 = sha256_file(WEIGHT_PATH)
if actual_predictor_sha256 != EXPECTED_PREDICTOR_SHA256:
    raise RuntimeError({"predictor_sha256": actual_predictor_sha256})
if actual_weight_sha256 != EXPECTED_WEIGHT_SHA256:
    raise RuntimeError({"weight_sha256": actual_weight_sha256})

print(json.dumps({
    "artifact_root": str(ARTIFACTS),
    "competition_dir": str(COMPETITION_DIR),
    "test_datasets": len(test_stems),
    "gpu_names": GPU_NAMES,
    "predictor_sha256": actual_predictor_sha256,
    "weight_sha256": actual_weight_sha256,
}, indent=2))


In [ ]:
RUNTIME_SOURCE = '#!/usr/bin/env python3\n"""Runtime template embedded in the top-k Kaggle production notebooks."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport csv\nimport hashlib\nimport json\nimport os\nimport sys\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport tracksdata as td\nfrom scipy.spatial import cKDTree\n\n\nSCALE_UM = np.asarray((1.625, 0.40625, 0.40625), dtype=np.float64)\nGRID_UM = 1.625\nDIVISION_INTERCEPT = -8.22818022705616\nDIVISION_COEFFICIENTS = np.asarray([\n    -0.11447652524963121, 1.7968352108138184, -0.7477483127548737,\n    -0.8527828000255034, -2.3768328955997426, -0.9436208184013919,\n    0.8690712572601186, 0.15578520308451954, 0.04972761045828509,\n    1.0321210246647057, -0.0735728543439014, 0.170043844145519,\n], dtype=np.float64)\nDIVISION_MEAN = np.asarray([\n    2.202441724390983, 11.905854115707049, 12.530247823741115,\n    5.888479769117546, 9.768692640375301, -0.1158564827485851,\n    0.1986536428846206, 0.9255220846423952, 0.2930777054607188,\n    -0.07684822100498105, 1.0423211975605249, 1.435286145506068,\n], dtype=np.float64)\nDIVISION_SCALE = np.asarray([\n    1.903330989651284, 2.4559407389543093, 2.8046694511907138,\n    1.494890787522451, 2.9891283923553686, 0.5657847102440112,\n    1.2386589875245562, 0.2625470538429257, 0.5500648796748153,\n    0.579434593739502, 0.22143015841180555, 0.5325751612102582,\n], dtype=np.float64)\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef import_predictor(repo: Path):\n    sys.path.insert(0, str(repo / "src"))\n    sys.path.insert(0, str(repo / "scripts"))\n    import predict_unet_transformer as predictor\n    return predictor\n\n\n@dataclass\nclass CandidateState:\n    seen_frames: set[int] = field(default_factory=set)\n    seen_pairs: set[tuple[int, int]] = field(default_factory=set)\n    coord_lists: list[np.ndarray] = field(default_factory=list)\n    coord_offset: dict[int, tuple[int, int]] = field(default_factory=dict)\n    global_node_count: int = 0\n    edges: list[tuple[int, int, float, float]] = field(default_factory=list)\n\n    def coords_so_far(self) -> np.ndarray:\n        if not self.coord_lists:\n            return np.empty((0, 4), dtype=np.int16)\n        return np.concatenate(self.coord_lists)\n\n\n@torch.no_grad()\ndef predict_video_topk(\n    predictor,\n    model,\n    ds_path: Path,\n    device: torch.device,\n    det_threshold: float,\n    window_size: int,\n    downsample: tuple[int, ...],\n    parents_per_target: int = 5,\n    max_edge_distance: float = 12.0,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:\n    """Exact model24 top-five candidate export for one detector threshold."""\n    if not 0.0 < det_threshold < 1.0:\n        raise ValueError("Invalid detector threshold")\n    ds = predictor.open_dataset(\n        ds_path, normalize=False, load_image=False, downsample=downsample\n    )\n    if "0.001" not in ds.quantiles or "0.999" not in ds.quantiles:\n        raise ValueError(f"Missing image quantiles: {ds_path}")\n    zarr_arr = predictor.zarr.open_group(str(ds.zarr_path), mode="r")["0"]\n    q_low = float(ds.quantiles["0.001"])\n    q_high = float(ds.quantiles["0.999"])\n    image_shape = ds.image_shape\n    target_shape = list(image_shape[1:])\n    ds_arr = np.asarray(downsample, dtype=np.float32)\n    ds_arr_t = torch.from_numpy(ds_arr).to(device)\n    voxel_size = tuple(scale * stride for scale, stride in zip(ds.scale, downsample))\n    pool_kernel = predictor.pool_kernel_from_um(3.0, voxel_size)\n    state = CandidateState()\n\n    stride = max(window_size - 1, 1)\n    window_starts = list(range(0, image_shape[0] - window_size + 1, stride))\n    if not window_starts or window_starts[-1] + window_size < image_shape[0]:\n        last = max(image_shape[0] - window_size, 0)\n        if not window_starts or last != window_starts[-1]:\n            window_starts.append(last)\n\n    for window_number, window_start in enumerate(window_starts, 1):\n        frame_indices = list(range(window_start, window_start + window_size))\n        frames = torch.stack([\n            predictor._load_frame(zarr_arr, frame, target_shape, downsample)\n            for frame in frame_indices\n        ])\n        frames = ((frames - q_low) / (q_high - q_low + 1e-6)).clamp(0.0)\n        images = frames.unsqueeze(0).to(device)\n        unet_out, det_logits = model.encode(images)\n        for dims in [(-1,), (-2,), (-2, -1)]:\n            images_flipped = images.flip(dims)\n            _, det_flipped = model.encode(images_flipped)\n            for local_frame in range(window_size):\n                det_logits[local_frame] += det_flipped[local_frame].flip(dims)\n            del images_flipped, det_flipped\n        for local_frame in range(window_size):\n            det_logits[local_frame] /= 4\n        del images, frames\n\n        for local_frame, frame in enumerate(frame_indices):\n            if frame in state.seen_frames:\n                continue\n            detected = predictor._detect_cells_pooled(\n                det_logits[local_frame][0],\n                frame,\n                det_threshold=det_threshold,\n                pool_kernel=pool_kernel,\n            )\n            state.coord_offset[frame] = (\n                state.global_node_count,\n                state.global_node_count + len(detected),\n            )\n            state.global_node_count += len(detected)\n            state.coord_lists.append(detected)\n            state.seen_frames.add(frame)\n\n        coords_so_far = state.coords_so_far()\n        for local_frame in range(window_size - 1):\n            source_t = frame_indices[local_frame]\n            target_t = frame_indices[local_frame + 1]\n            pair = (source_t, target_t)\n            if pair in state.seen_pairs:\n                continue\n            state.seen_pairs.add(pair)\n            source_start, source_end = state.coord_offset[source_t]\n            target_start, target_end = state.coord_offset[target_t]\n            if source_start == source_end or target_start == target_end:\n                continue\n            source_coords = coords_so_far[source_start:source_end]\n            target_coords = coords_so_far[target_start:target_end]\n            source_indices = np.arange(source_start, source_end, dtype=np.int64)\n            target_indices = np.arange(target_start, target_end, dtype=np.int64)\n            source_tensor = torch.from_numpy(\n                source_coords[:, 1:].astype(np.float32)\n            ).unsqueeze(0).to(device)\n            target_tensor = torch.from_numpy(\n                target_coords[:, 1:].astype(np.float32)\n            ).unsqueeze(0).to(device)\n            window_shape = (window_size,) + image_shape[1:]\n            source_relative = source_coords.copy()\n            target_relative = target_coords.copy()\n            source_relative[:, 0] = local_frame\n            target_relative[:, 0] = local_frame + 1\n            source_position = torch.from_numpy(\n                predictor.extract_pos_features(source_relative, window_shape)\n            ).unsqueeze(0).to(device)\n            target_position = torch.from_numpy(\n                predictor.extract_pos_features(target_relative, window_shape)\n            ).unsqueeze(0).to(device)\n            source_mask = torch.ones(\n                1, len(source_coords), dtype=torch.bool, device=device\n            )\n            target_mask = torch.ones(\n                1, len(target_coords), dtype=torch.bool, device=device\n            )\n            source_features = model._index_features(\n                unet_out[:, local_frame], source_tensor, source_mask\n            )\n            target_features = model._index_features(\n                unet_out[:, local_frame + 1], target_tensor, target_mask\n            )\n            raw = model.predict_edges(\n                source_features,\n                target_features,\n                source_tensor * ds_arr_t,\n                target_tensor * ds_arr_t,\n                source_position,\n                target_position,\n                source_mask,\n                target_mask,\n            )[0]\n            probabilities = torch.softmax(raw, dim=0)\n            k = min(parents_per_target, len(source_coords))\n            top_probabilities, top_sources = torch.topk(\n                probabilities, k=k, dim=0, sorted=True\n            )\n            top_probabilities = top_probabilities.cpu().numpy()\n            top_sources = top_sources.cpu().numpy()\n            candidates = [\n                (\n                    top_probabilities[rank, target_local],\n                    int(top_sources[rank, target_local]),\n                    target_local,\n                )\n                for target_local in range(len(target_coords))\n                for rank in range(k)\n            ]\n            candidates.sort(reverse=True)\n            for probability, source_local, target_local in candidates:\n                source_global = int(source_indices[source_local])\n                target_global = int(target_indices[target_local])\n                distance = float(\n                    np.linalg.norm(\n                        coords_so_far[source_global, 1:].astype(np.float32)\n                        - coords_so_far[target_global, 1:].astype(np.float32)\n                    )\n                )\n                if distance <= max_edge_distance:\n                    state.edges.append(\n                        (source_global, target_global, probability, distance)\n                    )\n        del unet_out, det_logits\n        if window_number % 10 == 0 or window_number == len(window_starts):\n            print(\n                f"{ds_path.stem}: window {window_number}/{len(window_starts)}",\n                flush=True,\n            )\n\n    coords = state.coords_so_far().astype(np.float32)\n    coords[:, 1:] *= ds_arr\n    return coords.astype(np.int16), state.edges\n\n\ndef run_worker() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--repo", type=Path, required=True)\n    parser.add_argument("--weights", type=Path, required=True)\n    parser.add_argument("--data-dir", type=Path, required=True)\n    parser.add_argument("--datasets-json", required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--det-threshold", type=float, default=0.965)\n    args = parser.parse_args()\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required for top-k inference")\n    datasets = json.loads(args.datasets_json)\n    if not isinstance(datasets, list) or not all(isinstance(item, str) for item in datasets):\n        raise ValueError("datasets-json must be a list of dataset stems")\n    predictor = import_predictor(args.repo)\n    device = torch.device("cuda")\n    model, window_size, downsample = predictor.load_model(args.weights, device)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    for number, stem in enumerate(datasets, 1):\n        destination = args.output_dir / f"{stem}.geff"\n        if destination.exists():\n            raise FileExistsError(destination)\n        print(\n            f"GPU {torch.cuda.get_device_name(0)}: {number}/{len(datasets)} {stem}",\n            flush=True,\n        )\n        coords, edges = predict_video_topk(\n            predictor,\n            model,\n            args.data_dir / stem,\n            device,\n            args.det_threshold,\n            window_size,\n            downsample,\n        )\n        graph = predictor.build_graph(coords, edges)\n        predictor.save_graph(graph, destination)\n        print(\n            json.dumps(\n                {\n                    "dataset": stem,\n                    "nodes": graph.num_nodes(),\n                    "candidate_edges": graph.num_edges(),\n                }\n            ),\n            flush=True,\n        )\n        torch.cuda.empty_cache()\n\n\n@contextlib.contextmanager\ndef suppress_output():\n    with open(os.devnull, "w") as devnull:\n        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):\n            yield\n\n\ndef load_graph(path: Path):\n    loaded = td.graph.IndexedRXGraph.from_geff(path)\n    return loaded[0] if isinstance(loaded, tuple) else loaded\n\n\ndef solve_topk_graph(path: Path, edge_floor: float = 0.40):\n    graph = load_graph(path)\n    edge_rows = list(\n        graph.edge_attrs(attr_keys=["edge_id", "edge_prob"])\n        .select("edge_id", "edge_prob")\n        .iter_rows(named=True)\n    )\n    remove_ids = [\n        int(row["edge_id"])\n        for row in edge_rows\n        if float(row["edge_prob"]) < edge_floor\n    ]\n    if remove_ids:\n        graph.bulk_remove_edges(remove_ids)\n    solver = td.solvers.ILPSolver(\n        edge_weight=-1.0 * td.EdgeAttr("edge_prob"),\n        appearance_weight=0.0,\n        disappearance_weight=2.0,\n        division_weight=1.2,\n        num_threads=1,\n        gap=0.0,\n    )\n    with suppress_output():\n        solution = solver.solve(graph)\n    return solution, {\n        "candidate_edges": len(edge_rows),\n        "retained_edges": len(edge_rows) - len(remove_ids),\n        "selected_edges": solution.num_edges(),\n    }\n\n\ndef close_internal_gaps(graph):\n    """Model30\'s fixed conservative gap rule, applied in memory."""\n    node_rows = list(\n        graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])\n        .select("node_id", "t", "z", "y", "x")\n        .iter_rows(named=True)\n    )\n    node_ids = np.asarray([int(row["node_id"]) for row in node_rows])\n    times = np.asarray([int(row["t"]) for row in node_rows])\n    coords = {\n        int(row["node_id"]): np.asarray(\n            (row["z"], row["y"], row["x"]), dtype=np.float64\n        )\n        * SCALE_UM\n        for row in node_rows\n    }\n    edge_rows = list(\n        graph.edge_attrs(attr_keys=["source_id", "target_id"])\n        .select("source_id", "target_id")\n        .iter_rows(named=True)\n    )\n    predecessor = {\n        int(row["target_id"]): int(row["source_id"]) for row in edge_rows\n    }\n    successor = {\n        int(row["source_id"]): int(row["target_id"]) for row in edge_rows\n    }\n    indegree = Counter(int(row["target_id"]) for row in edge_rows)\n    outdegree = Counter(int(row["source_id"]) for row in edge_rows)\n    additions = []\n    candidate_count = 0\n    inclusive_radius_um = np.nextafter(5.0 * GRID_UM, np.inf)\n    for target_time in sorted(set(times.tolist())):\n        source_ids = np.asarray(\n            [\n                int(node)\n                for node in node_ids[times == target_time - 1]\n                if outdegree[int(node)] == 0\n            ]\n        )\n        target_ids = np.asarray(\n            [\n                int(node)\n                for node in node_ids[times == target_time]\n                if indegree[int(node)] == 0\n            ]\n        )\n        if len(source_ids) == 0 or len(target_ids) == 0:\n            continue\n        source_coords = np.asarray([coords[int(node)] for node in source_ids])\n        target_coords = np.asarray([coords[int(node)] for node in target_ids])\n        distances, source_indices = cKDTree(source_coords).query(\n            target_coords,\n            k=1,\n            distance_upper_bound=inclusive_radius_um,\n            workers=-1,\n        )\n        frame_candidates = []\n        for target_index, (distance, source_index) in enumerate(\n            zip(distances, source_indices, strict=True)\n        ):\n            if np.isfinite(distance) and source_index < len(source_ids):\n                frame_candidates.append(\n                    (\n                        int(source_ids[int(source_index)]),\n                        int(target_ids[target_index]),\n                        float(distance),\n                    )\n                )\n        candidate_count += len(frame_candidates)\n        claimed_sources: set[int] = set()\n        for source, target, distance in sorted(\n            frame_candidates, key=lambda row: (row[0], row[2], row[1])\n        ):\n            if source in claimed_sources:\n                continue\n            claimed_sources.add(source)\n            previous = predecessor.get(source)\n            following = successor.get(target)\n            if previous is None or following is None:\n                continue\n            vector = coords[target] - coords[source]\n            previous_vector = coords[source] - coords[previous]\n            following_vector = coords[following] - coords[target]\n            acceleration = min(\n                float(np.linalg.norm(vector - previous_vector)),\n                float(np.linalg.norm(following_vector - vector)),\n            )\n            if acceleration > 6.5:\n                continue\n            additions.append(\n                {\n                    "source_id": source,\n                    "target_id": target,\n                    "edge_prob": 1.0,\n                    "edge_dist": distance / GRID_UM,\n                }\n            )\n    if additions:\n        graph.bulk_add_edges(additions)\n    return {"candidate_gaps": candidate_count, "edges_added": len(additions)}\n\n\ndef _norm(vector: np.ndarray) -> float:\n    return float(np.linalg.norm(vector))\n\n\ndef _cosine(left: np.ndarray, right: np.ndarray) -> float:\n    denominator = _norm(left) * _norm(right)\n    return float(np.dot(left, right) / denominator) if denominator > 1e-9 else 0.0\n\n\ndef _division_probability(features: tuple[float, ...]) -> float:\n    logit = (\n        DIVISION_INTERCEPT\n        + ((np.asarray(features) - DIVISION_MEAN) / DIVISION_SCALE)\n        @ DIVISION_COEFFICIENTS\n    )\n    return float(1.0 / (1.0 + np.exp(-np.clip(logit, -40.0, 40.0))))\n\n\ndef add_sparse_division_rescue(graph):\n    """Fixed model42 gate; at most one fork in an otherwise fork-free graph."""\n    node_rows = list(\n        graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])\n        .select("node_id", "t", "z", "y", "x")\n        .iter_rows(named=True)\n    )\n    nodes = {\n        int(row["node_id"]): (\n            int(row["t"]),\n            np.asarray((row["z"], row["y"], row["x"]), dtype=np.float64)\n            * SCALE_UM,\n        )\n        for row in node_rows\n    }\n    edge_rows = list(\n        graph.edge_attrs(attr_keys=["source_id", "target_id"])\n        .select("source_id", "target_id")\n        .iter_rows(named=True)\n    )\n    successors: dict[int, list[int]] = defaultdict(list)\n    predecessors: dict[int, list[int]] = defaultdict(list)\n    for row in edge_rows:\n        source, target = int(row["source_id"]), int(row["target_id"])\n        successors[source].append(target)\n        predecessors[target].append(source)\n    if any(len(children) >= 2 for children in successors.values()):\n        return {\n            "status": "skipped_existing_fork",\n            "candidates_scored": 0,\n            "candidates_passing_gate": 0,\n            "edges_added": 0,\n        }\n\n    by_time: dict[int, list[int]] = defaultdict(list)\n    for node_id, (time, _) in nodes.items():\n        by_time[time].append(node_id)\n    proposals: list[tuple[float, int, int, float]] = []\n    candidates_scored = 0\n    for time in sorted(by_time):\n        source_ids = [\n            node_id\n            for node_id in by_time[time]\n            if len(successors[node_id]) == 1 and len(predecessors[node_id]) == 1\n        ]\n        orphan_ids = [\n            node_id\n            for node_id in by_time.get(time + 1, [])\n            if len(predecessors[node_id]) == 0\n        ]\n        if not source_ids or not orphan_ids:\n            continue\n        frame_ids = by_time[time + 1]\n        frame_positions = np.stack([nodes[node_id][1] for node_id in frame_ids])\n        orphan_positions = np.stack([nodes[node_id][1] for node_id in orphan_ids])\n        for source_id in source_ids:\n            linked_id = successors[source_id][0]\n            if len(successors[linked_id]) != 1:\n                continue\n            parent = nodes[source_id][1]\n            linked = nodes[linked_id][1]\n            parent_linked = _norm(linked - parent)\n            if parent_linked > 12.0:\n                continue\n            nearest = orphan_ids[\n                int(np.argmin(np.linalg.norm(orphan_positions - linked, axis=1)))\n            ]\n            previous_velocity = parent - nodes[predecessors[source_id][0]][1]\n            candidate_id = nearest\n            if len(successors[candidate_id]) != 1:\n                continue\n            candidate = nodes[candidate_id][1]\n            parent_candidate = _norm(candidate - parent)\n            if parent_candidate > 15.0:\n                continue\n            sister = _norm(candidate - linked)\n            if sister > 20.5:\n                continue\n            next_linked = nodes[successors[linked_id][0]][1]\n            next_candidate = nodes[successors[candidate_id][0]][1]\n            separation_growth = _norm(next_linked - next_candidate) - sister\n            if separation_growth < 0.0:\n                continue\n            positive_distances = np.sort(\n                np.linalg.norm(frame_positions - linked, axis=1)\n            )\n            positive_distances = positive_distances[positive_distances > 1e-9]\n            rank = 1 + int(np.sum(positive_distances < sister - 1e-9))\n            local_count = int(\n                np.sum(np.linalg.norm(frame_positions - parent, axis=1) <= 12.0)\n            )\n            features = (\n                parent_linked,\n                parent_candidate,\n                sister,\n                _norm((linked + candidate) * 0.5 - parent),\n                abs(parent_linked - parent_candidate),\n                _cosine(linked - parent, candidate - parent),\n                separation_growth,\n                1.0,\n                _cosine(previous_velocity, linked - parent),\n                _cosine(previous_velocity, candidate - parent),\n                float(rank),\n                float(local_count),\n            )\n            candidates_scored += 1\n            probability = _division_probability(features)\n            if not (\n                0.45 <= probability < 0.5668243328192077\n                and parent_candidate <= 8.0\n                and sister <= 13.0\n                and features[3] <= 3.0\n                and features[4] <= 3.5\n                and features[5] <= -0.5\n                and separation_growth >= 1.0\n                and rank <= 2\n                and local_count <= 2\n            ):\n                continue\n            proposals.append((probability, source_id, candidate_id, parent_candidate))\n    if not proposals:\n        return {\n            "status": "no_candidate",\n            "candidates_scored": candidates_scored,\n            "candidates_passing_gate": 0,\n            "edges_added": 0,\n        }\n    probability, source, target, distance = sorted(\n        proposals, key=lambda row: (-row[0], row[1], row[2])\n    )[0]\n    if len(successors[source]) != 1 or len(predecessors[target]) != 0:\n        return {\n            "status": "candidate_no_longer_valid",\n            "candidates_scored": candidates_scored,\n            "candidates_passing_gate": len(proposals),\n            "edges_added": 0,\n        }\n    graph.bulk_add_edges(\n        [\n            {\n                "source_id": source,\n                "target_id": target,\n                "edge_prob": 1.0,\n                "edge_dist": distance / GRID_UM,\n            }\n        ]\n    )\n    return {\n        "status": "added",\n        "candidates_scored": candidates_scored,\n        "candidates_passing_gate": len(proposals),\n        "edges_added": 1,\n        "source_id": source,\n        "target_id": target,\n        "probability": probability,\n    }\n\n\ndef graph_receipt(graph, dataset: str) -> dict[str, int | str]:\n    edges = list(\n        graph.edge_attrs(attr_keys=["source_id", "target_id"])\n        .select("source_id", "target_id")\n        .iter_rows()\n    )\n    indegree = Counter(int(target) for _, target in edges)\n    outdegree = Counter(int(source) for source, _ in edges)\n    max_in = max(indegree.values(), default=0)\n    max_out = max(outdegree.values(), default=0)\n    if max_in > 1 or max_out > 2:\n        raise RuntimeError(f"{dataset}: invalid degrees {max_in}/{max_out}")\n    return {\n        "dataset": dataset,\n        "nodes": graph.num_nodes(),\n        "edges": len(edges),\n        "divisions": sum(value == 2 for value in outdegree.values()),\n        "max_indegree": max_in,\n        "max_outdegree": max_out,\n    }\n\n\ndef write_submission(graphs: dict[str, object], test_stems: list[str], output: Path):\n    columns = [\n        "id",\n        "dataset",\n        "row_type",\n        "node_id",\n        "t",\n        "z",\n        "y",\n        "x",\n        "source_id",\n        "target_id",\n    ]\n    if sorted(graphs) != sorted(test_stems):\n        raise RuntimeError({"graphs": sorted(graphs), "test_stems": sorted(test_stems)})\n    next_id = 0\n    receipts = []\n    with output.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, lineterminator="\\n")\n        writer.writeheader()\n        for dataset in sorted(test_stems):\n            graph = graphs[dataset]\n            node_rows = list(\n                graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])\n                .select("node_id", "t", "z", "y", "x")\n                .iter_rows(named=True)\n            )\n            edge_rows = list(\n                graph.edge_attrs(attr_keys=["source_id", "target_id"])\n                .select("source_id", "target_id")\n                .iter_rows(named=True)\n            )\n            node_ids = {int(row["node_id"]) for row in node_rows}\n            node_times = {\n                int(row["node_id"]): int(row["t"]) for row in node_rows\n            }\n            pairs: set[tuple[int, int]] = set()\n            for row in edge_rows:\n                source, target = int(row["source_id"]), int(row["target_id"])\n                if source not in node_ids or target not in node_ids:\n                    raise RuntimeError(f"{dataset}: dangling edge")\n                if node_times[target] != node_times[source] + 1:\n                    raise RuntimeError(f"{dataset}: nonconsecutive edge")\n                if (source, target) in pairs:\n                    raise RuntimeError(f"{dataset}: duplicate edge")\n                pairs.add((source, target))\n            for row in node_rows:\n                writer.writerow(\n                    {\n                        "id": next_id,\n                        "dataset": dataset,\n                        "row_type": "node",\n                        "node_id": int(row["node_id"]),\n                        "t": int(row["t"]),\n                        "z": max(0, int(round(float(row["z"])))),\n                        "y": max(0, int(round(float(row["y"])))),\n                        "x": max(0, int(round(float(row["x"])))),\n                        "source_id": -1,\n                        "target_id": -1,\n                    }\n                )\n                next_id += 1\n            for row in edge_rows:\n                writer.writerow(\n                    {\n                        "id": next_id,\n                        "dataset": dataset,\n                        "row_type": "edge",\n                        "node_id": -1,\n                        "t": -1,\n                        "z": -1,\n                        "y": -1,\n                        "x": -1,\n                        "source_id": int(row["source_id"]),\n                        "target_id": int(row["target_id"]),\n                    }\n                )\n                next_id += 1\n            receipts.append(graph_receipt(graph, dataset))\n    with output.open(newline="", encoding="utf-8") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != columns:\n            raise RuntimeError("Bad CSV schema")\n        rows = list(reader)\n    if any(int(row["id"]) != number for number, row in enumerate(rows)):\n        raise RuntimeError("Nonconsecutive CSV IDs")\n    if {row["dataset"] for row in rows} != set(test_stems):\n        raise RuntimeError("CSV dataset set mismatch")\n    return {"rows": next_id, "datasets": receipts, "sha256": sha256_file(output)}\n\n\nif __name__ == "__main__":\n    run_worker()\n'
# This embedded file is checksummed before the GPU workers launch.


In [ ]:
RUNTIME_PATH = WORKING_DIR / "topk_runtime.py"
RUNTIME_PATH.write_text(RUNTIME_SOURCE)
RUNTIME_SHA256 = sha256_file(RUNTIME_PATH)

candidate_root = WORKING_DIR / "model_topk_candidates"
if candidate_root.exists():
    shutil.rmtree(candidate_root)
candidate_root.mkdir(parents=True)
chunks = [test_stems[index::GPU_COUNT] for index in range(GPU_COUNT)]
if any(not chunk for chunk in chunks):
    raise RuntimeError({"test_datasets": len(test_stems), "gpu_count": GPU_COUNT})
started = time.monotonic()
processes: list[tuple[list[str], subprocess.Popen]] = []
for gpu_index, chunk in enumerate(chunks):
    output_dir = candidate_root / f"gpu_{gpu_index}"
    command = [
        sys.executable, str(RUNTIME_PATH), "--repo", str(REPO_DIR),
        "--weights", str(WEIGHT_PATH), "--data-dir", str(TEST_DIR),
        "--datasets-json", json.dumps(chunk), "--output-dir", str(output_dir),
        "--det-threshold", str(DET_THRESHOLD),
    ]
    environment = {
        **os.environ,
        "CUDA_VISIBLE_DEVICES": str(gpu_index),
        "PYTHONPATH": f"{REPO_DIR / 'src'}:{REPO_DIR / 'scripts'}",
        "OMP_NUM_THREADS": "2",
    }
    print("Launching:", " ".join(command), flush=True)
    processes.append((command, subprocess.Popen(command, cwd=WORKING_DIR, env=environment)))
failures = []
for command, process in processes:
    returncode = process.wait()
    if returncode:
        failures.append({"returncode": returncode, "command": command})
if failures:
    raise RuntimeError({"worker_failures": failures})
candidate_paths: dict[str, Path] = {}
for graph_path in sorted(candidate_root.glob("gpu_*/*.geff")):
    if graph_path.stem in candidate_paths:
        raise RuntimeError(f"Duplicate candidate graph: {graph_path.stem}")
    candidate_paths[graph_path.stem] = graph_path
if sorted(candidate_paths) != test_stems:
    raise RuntimeError({
        "missing": sorted(set(test_stems) - set(candidate_paths)),
        "extra": sorted(set(candidate_paths) - set(test_stems)),
    })
inference_seconds = time.monotonic() - started
print(f"Top-five candidate inference completed in {inference_seconds / 60.0:.2f} minutes")


In [ ]:
import importlib

sys.path.insert(0, str(WORKING_DIR))
runtime = importlib.import_module("topk_runtime")
ENABLE_SPARSE_RESCUE = False
output_dir = WORKING_DIR / "model_topk_processed"
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True)
graphs: dict[str, object] = {}
dataset_receipts = []
postprocess_started = time.monotonic()
for number, dataset in enumerate(test_stems, 1):
    graph, solve_receipt = runtime.solve_topk_graph(
        candidate_paths[dataset], edge_floor=0.40
    )
    gap_receipt = runtime.close_internal_gaps(graph)
    sparse_receipt = (
        runtime.add_sparse_division_rescue(graph)
        if ENABLE_SPARSE_RESCUE
        else {
            "status": "disabled",
            "candidates_scored": 0,
            "candidates_passing_gate": 0,
            "edges_added": 0,
        }
    )
    structural = runtime.graph_receipt(graph, dataset)
    graph.to_geff(output_dir / f"{dataset}.geff")
    graphs[dataset] = graph
    dataset_receipts.append({
        **structural,
        "solver": solve_receipt,
        "gap_close": gap_receipt,
        "sparse_rescue": sparse_receipt,
    })
    print(
        f"{number}/{len(test_stems)} {dataset}: "
        f"{json.dumps(dataset_receipts[-1], sort_keys=True)}",
        flush=True,
    )
submission_path = WORKING_DIR / "submission.csv"
csv_receipt = runtime.write_submission(graphs, test_stems, submission_path)
receipt = {
    "status": "complete",
    "model": "model56",
    "ground_truth_accessed": False,
    "test_datasets": len(test_stems),
    "runtime_sha256": RUNTIME_SHA256,
    "predictor_sha256": actual_predictor_sha256,
    "weight_sha256": actual_weight_sha256,
    "inference_seconds": inference_seconds,
    "postprocess_seconds": time.monotonic() - postprocess_started,
    "gpu_names": GPU_NAMES,
    "config": {
        "det_threshold": DET_THRESHOLD,
        "parents_per_target": 5,
        "candidate_max_distance_grid": 12.0,
        "edge_probability_floor": 0.40,
        "ilp": {
            "edge_weight": -1.0,
            "appearance_weight": 0.0,
            "disappearance_weight": 2.0,
            "division_weight": 1.2,
        },
        "gap_close": {
            "radius_grid": 5.0,
            "min_acceleration_um": 6.5,
            "require_internal": True,
        },
        "sparse_division_rescue": ENABLE_SPARSE_RESCUE,
    },
    "csv": csv_receipt,
    "datasets": dataset_receipts,
}
receipt_path = WORKING_DIR / "model56_runtime_receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n")
print(json.dumps(receipt, indent=2, sort_keys=True))
print(f"Wrote {submission_path} ({submission_path.stat().st_size:,} bytes)")
